# FutWAR: first player-metrics exploration

User-selected focus: attacking, progression and defensive distributions by nominal position. Main displays use **900+ minutes**, with exclusions counted. This is an exploratory display choice, not a replacement baseline or model eligibility rule.

Grain: one player-team-season source record. Transfers are not combined into player seasons. All eight seasons, including COVID years, remain. Nominal labels are retained exactly; no tactical roles or new position families are assigned. No value score, WAR formula or league-strength rating is estimated.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Image

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'AGENTS.md').exists(), 'Run from the FutWAR project or notebooks folder.'
sys.path.insert(0, str(ROOT / 'scripts'))
from explore_player_metrics import run, METRICS, LABELS, eligibility_masks, weighted_rate

MIN_MINUTES = 900  # Explicit user choice for these exploratory displays.
players, exclusions, distributions, context, figures, summary = run(ROOT, MIN_MINUTES)
display(pd.Series({k: v for k, v in summary.items() if k not in ['snapshot', 'position_labels']}))

## 1. Who is included?

The 900-minute cutoff is applied to each club stint, not a verified player's combined clubs. A transfer can therefore split exposure across records. Goalkeepers are counted separately because the project's initial contribution method targets outfield players. Missing metrics remain missing; zeros remain observed zeros.

In [ ]:
display(exclusions)
display(exclusions[['all_records','outfield_records','eligible_records','outfield_below_cutoff','goalkeeper_records','unknown_position_records','outfield_missing_minutes']].sum().rename('records'))
outfield, eligible = eligibility_masks(players, MIN_MINUTES)
display(players.loc[eligible].groupby('nominal_position').agg(records=('output_record','size'), total_minutes=('minutes','sum')))

## 2. Metric meanings and rate calculations

These eight fields follow the existing source-total mappings in `sql/005_exploration_views.sql`. Exact historical provider definitions still need confirmation before modeling. [Sports Reference documentation](https://info.sports-reference.com/migration/knowledge/soccer-advanced-stats-explained) distinguishes xA and xAG; they remain separate here. Do not add related expected or progression measures together as independent contributions.

Individual per-90 = **90 × metric total / minutes**. Exposure-weighted group per-90 = **90 × sum(known metric totals) / sum(minutes for those same metric-known records)**. The median describes a typical record; the weighted rate reflects aggregate exposure. Neither adjusts for possession, teammates, opponents, tactical role or league quality.

In [ ]:
display(distributions[['league','season','nominal_position','metric','eligible_records','eligible_metric_known_records','eligible_missing_metric_records','known_metric_minutes','median_per90','exposure_weighted_per90']].head(24))
display(distributions.groupby('metric')['eligible_missing_metric_records'].sum().rename('missing cells in selected cohort'))

## 3. Position distributions

The box shows the middle half of records and the median; dots show values outside the plot's whiskers. All original mixed labels are displayed separately, including order differences such as MF,FW and FW,MF. The underlying CSV includes 5th/25th/50th/75th/95th percentiles, minima, maxima and counts for every league × season × observed outfield position × metric combination.

This figure pools leagues and seasons. Differences can reflect composition and opportunity, not greater player contribution. Read the sample counts alongside it.

In [ ]:
display(Image(filename=str(figures[0])))
pooled = []
for position, group in players.loc[eligible].groupby('nominal_position'):
    for metric in METRICS:
        rates = 90 * group[metric] / group['minutes']
        pooled.append(dict(position=position, metric=metric, known_records=rates.notna().sum(),
                           median_per90=rates.median(), exposure_weighted_per90=weighted_rate(group[metric], group['minutes'])))
display(pd.DataFrame(pooled).pivot(index='position', columns='metric', values='median_per90'))

## 4. League and season trends

These are exposure-weighted rates for exact DF, MF and FW source labels. The full summary still contains every mixed label. Membership changes across seasons: a rise may reflect who qualified for the cutoff, position coding, opportunity or statistical definitions. It does not establish a COVID effect or absolute league strength.

In [ ]:
for path in figures[1:4]:
    display(Image(filename=str(path)))

## 5. Schedule and COVID context

This separate exposure figure uses **all player records, including goalkeepers**, to inspect the schedule. A lower season total can result from fewer completed games or fewer clubs. Minutes per completed team-game helps distinguish schedule volume from the amount recorded per game. No COVID seasons are excluded, rescaled or assigned a causal effect.

In [ ]:
display(Image(filename=str(figures[4])))
display(context.loc[context.league.eq('Ligue 1'), ['season','team_labels','team_games','player_minutes_per_team','player_minutes_per_team_game','player_minutes_per_team_change_pct']])

## 6. Small groups and open research questions

Empty groups remain explicit with missing rates. Sort by sample size to identify comparisons needing caution; no new sample-size exclusion is imposed.

Discuss before the next modeling step:
- Should mixed position labels be combined, and what evidence would support that mapping?
- How sensitive are comparisons to the 900-minute display cutoff and transferred club stints?
- Which metric definitions and context adjustments need further verification?
- Should COVID treatment use explicit league/season context or a sensitivity analysis?

Replacement definitions, weights, cross-league strength calibration and tactical roles remain open. Keep xA/xAG and correlated progression measures distinct until their overlap is studied.

In [ ]:
display(distributions[['league','season','nominal_position','metric','eligible_metric_known_records']].sort_values('eligible_metric_known_records').head(32))
print('Outputs:', ROOT / 'reports/exploration')
print('No research formula or replacement baseline was applied.')